# Data Lakehouse — Pipeline de Datos de Fútbol

Este notebook ejecuta el pipeline completo de integración de datos de jugadores, equipos y ligas a partir de tres fuentes: **Transfermarkt**, **TheSportsDB** y **FBRef**.

**Secciones:**
1. [Ingesta](#1.-Ingesta) — Conexión a S3, inicialización de Spark e ingesta de datos bronce.
2. [ID Matching](#2.-ID-Matching) — Cruce fuzzy de entidades entre fuentes.
3. [Limpieza y Merge](#3.-Limpieza-y-Merge) — Consolidación, validación y escritura de tablas finales.

---
## 1. Ingesta

Instalación de dependencias, conexión al bucket S3 y arranque de la sesión Spark. A continuación se ejecuta la ingesta hacia la capa **bronce** para cada una de las tres fuentes de datos.

### 1.1 Dependencias y conexión a S3

In [1]:
!pip install pyspark
!sudo apt-get update && sudo apt-get install -y openjdk-17-jdk
!python3 -m pip install soccerdata
!java -version
!sudo pip install awscli
!sudo pip install boto3
!sudo pip install rapidfuzz
!sudo pip install pyiceberg
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://172.16.58.11:31224",
    aws_access_key_id="GK5f421d5f440758f74b0e0312",
    aws_secret_access_key="409baa63477885db12cd1db0a518748c5e83e971b5e8cf2129fe6c7498de125d",
    region_name="us-east-1"
)

# Listar buckets
response = s3.list_buckets()
for bucket in response["Buckets"]:
    print(bucket["Name"])

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk is already the newest version (17.0.19+10-1~22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 151 not upgraded.
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
Using cached rich-15.0.0-py3-none-any.whl (310 kB)
  Attempting uninstall: rich
    Found existing installation: rich 14.3.4
    Uninstalling rich-14.3.4:
      Successfully uninstalled rich-14.3.4
  Rolling back uninstall of rich
  Moving to /home/jovyan/.local/lib/python3.11/site-packages/rich-14.3.4.dist-info/
   from /home/jovyan/.local/lib/python3.11/site-packages/~ich-14.3.4.dist-info
  Moving to /home/jovyan/.local/lib/

### 1.2 Inicialización de Spark

In [2]:
import subprocess
subprocess.run(["pip", "install", "--user", "pyiceberg[s3fsspec,pyarrow]"], check=True)

# 1. Parar sesión existente
from pyspark.sql import SparkSession
active = SparkSession.getActiveSession()
if active:
    active.stop()

import sys
sys.path.insert(0, '/home/jovyan/work/Data-Lakehouse')
import ingesta
# 3. Crear sesión nueva
spark = ingesta.get_spark()
print (spark.version)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8746d432-73db-43b7-99af-4cd0041ec0e0;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in spark-list
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in spark-list
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in spark-list
	found com.amazonaws#aws-java-sdk-bundle;1.12.367 in central
:: resolution rep

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
4.1.2


### 1.3 Ingesta por fuente

Se ejecuta `run_ingesta` para cada fuente. Los datos se almacenan en la capa bronce del lakehouse.

In [ ]:
spark.sql(f"SHOW NAMESPACES in players").show()

In [ ]:
spark.sql(f"DROP NAMESPACE players.mock").show()

In [ ]:
import thesportsdb_bronce
ingesta.run_ingesta(thesportsdb_bronce, test_mode=True, namespace="mock")

In [ ]:
import transfermarkt_bronce
ingesta.run_ingesta(transfermarkt_bronce, test_mode=True, namespace="mock")

In [ ]:
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--user", "soccerdata"],
    check=True
)
import fbref_bronce

fbref_bronce.get_data(namespace="mock", test_mode=True)

---
## 2. ID Matching

Cruce de identidades entre las tres fuentes mediante similitud fuzzy (`rapidfuzz`).
En el notebook se usa el mismo flujo que la UI para garantizar resultados consistentes:
1. `generar_mapeo_df` (pandas)
2. Guardar mapeo en Iceberg
3. `unir_fuentes_df` (pandas)
4. Guardar tabla unificada en Iceberg

La UI queda como apoyo visual para configurar y validar; la ejecución de referencia para la presentación es este notebook.

### 2.1 Jugadores

In [3]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], check=True)
import subprocess
subprocess.run(["pip", "install", "--user", "pyiceberg[s3fsspec,pyarrow]"], check=True)

import importlib
import IDMatching
importlib.reload(IDMatching)
from ingesta import get_catalog

sources = [
    {"table": "mock.transfermarkt_players", "name_col": "name",      "date_col": "dateOfBirth", "id_col": "id",       "prefix": "tm", "key_mode": "full_date", "umbral": 88},
    {"table": "mock.thesportsdb_players",   "name_col": "strPlayer", "date_col": "dateBorn",    "id_col": "idPlayer", "prefix": "ts", "key_mode": "full_date", "umbral": 88},
    {"table": "mock.fbref_players",         "name_col": "player",    "date_col": "born",        "id_col": None,       "prefix": "fb", "key_mode": "year",      "umbral": 75},
]

mapeo, tabla_final = IDMatching.run_matching(
    sources       = sources,
    catalog       = get_catalog(),
    umbral        = 85,
    mapping_table = "players.mock.players_mapping",
    unified_table = "players.mock.players_unified",
)

print("Mapeo:      ", len(mapeo))
print("Unificado:  ", len(tabla_final))

/home/jovyan/.local/lib/python3.11/site-packages/pyiceberg/table/__init__.py:639: UserWarning: Delete operation did not match any records
  self.delete(



Total filas en tabla unificada: 182
  Con datos de tm: 75 (41.2%)
  Con datos de ts: 75 (41.2%)
  Con datos de fb: 38 (20.9%)


/home/jovyan/.local/lib/python3.11/site-packages/pyiceberg/table/__init__.py:639: UserWarning: Delete operation did not match any records
  self.delete(


Mapeo:       181
Unificado:   182


In [ ]:
import importlib
import pandas as pd
import IDMatching
importlib.reload(IDMatching)

# 1) Cargar a pandas
df_tm = spark.table("players.mock.transfermarkt_players").toPandas()
df_ts = spark.table("players.mock.thesportsdb_players").toPandas()
df_fbref = spark.table("players.mock.fbref_players").toPandas()

# 2) Mapeo en pandas (igual que UI)
mapeo_pd = IDMatching.generar_mapeo_df(
    (df_tm,    "name",      "dateOfBirth", "id",       "tm", IDMatching.clave_fecha_completa, 88),
    (df_ts,    "strPlayer", "dateBorn",    "idPlayer", "ts", IDMatching.clave_fecha_completa, 88),
    (df_fbref, "player",    "born",        None,       "fb", IDMatching.clave_solo_anio,      75),
    umbral=85
)

# 3) Preparar FBref con clave derivada para el join (igual concepto que UI)
df_fbref_join = df_fbref.copy()
df_fbref_join["_clave"] = df_fbref_join.apply(
    lambda r: IDMatching.clave_solo_anio(r.get("player"), r.get("born")),
    axis=1
)

# 4) Unificación en pandas (igual que UI)
tabla_final_pd = IDMatching.unir_fuentes_df(
    mapeo_pd,
    (df_tm,        "id",       "tm"),
    (df_ts,        "idPlayer", "ts"),
    (df_fbref_join,"_clave",   "fb"),
)

# 5) Guardar en Iceberg (Spark solo al final, no entre paso 2 y 4)
spark.createDataFrame(mapeo_pd).writeTo("players.mock.players_mapping").using("iceberg").createOrReplace()
spark.createDataFrame(tabla_final_pd).writeTo("players.mock.players_unified").using("iceberg").createOrReplace()

print("Mapeo:       ", len(mapeo_pd))
print("Tabla final: ", len(tabla_final_pd))
print("Filas tm:    ", len(df_tm))

### 2.2 Equipos

In [ ]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import importlib
import IDMatching
importlib.reload(IDMatching)

# Para teams
df_tm = spark.table("players.mock.transfermarkt_teams").toPandas()
df_ts = spark.table("players.mock.thesportsdb_teams").toPandas()
df_fbref = spark.table("players.mock.fbref_teams").toPandas()

# Paso 1: generar mapeo (pandas, igual que UI)
mapeo_pd = IDMatching.generar_mapeo_df(
    (df_tm,    "name",      "foundedOn",      "id",      "tm", IDMatching.clave_equipo_fecha,     88),
    (df_ts,    "strTeam",   "intFormedYear",  "idTeam",  "ts", IDMatching.clave_equipo_fecha,     88),
    (df_fbref, "team",      None,             None,      "fb", IDMatching.clave_equipo_solo_anio, 75),
    umbral=85
)

# Paso 2: guardar mapeo
spark.createDataFrame(mapeo_pd) \
    .writeTo("players.mock.teams_mapping") \
    .using("iceberg").createOrReplace()

# Paso 3: preparar clave derivada para fbref y unir
df_fbref_join = df_fbref.copy()
df_fbref_join["_clave"] = df_fbref_join.apply(
    lambda r: IDMatching.clave_equipo_solo_anio(r.get("team"), None),
    axis=1
)

tabla_final_pd = IDMatching.unir_fuentes_df(
    mapeo_pd,
    (df_tm,         "id",      "tm"),
    (df_ts,         "idTeam",  "ts"),
    (df_fbref_join, "_clave",  "fb"),
)

# Paso 4: guardar tabla final
spark.createDataFrame(tabla_final_pd) \
    .writeTo("players.mock.teams_unified") \
    .using("iceberg").createOrReplace()

print("Mapeo:       ", len(mapeo_pd))
print("Tabla final: ", len(tabla_final_pd))
print("Filas tm:    ", len(df_tm))

### 2.3 Ligas

In [ ]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import importlib
import IDMatching
importlib.reload(IDMatching)

# Para leagues
df_tm = spark.table("players.mock.transfermarkt_leagues").toPandas()
df_ts = spark.table("players.mock.thesportsdb_leagues").toPandas()

# Paso 1: generar mapeo (pandas, igual que UI)
mapeo_pd = IDMatching.generar_mapeo_df(
    (df_tm, "name",      "country",    "id",       "tm", IDMatching.clave_liga, 88),
    (df_ts, "strLeague", "strCountry", "idLeague", "ts", IDMatching.clave_liga, 88),
    umbral=85
)

# Paso 2: guardar mapeo
spark.createDataFrame(mapeo_pd) \
    .writeTo("players.mock.leagues_mapping") \
    .using("iceberg").createOrReplace()

# Paso 3: unir datos
tabla_final_pd = IDMatching.unir_fuentes_df(
    mapeo_pd,
    (df_tm, "id",       "tm"),
    (df_ts, "idLeague", "ts"),
)

# Paso 4: guardar tabla final
spark.createDataFrame(tabla_final_pd) \
    .writeTo("players.mock.leagues_unified") \
    .using("iceberg").createOrReplace()

print("Mapeo:       ", len(mapeo_pd))
print("Tabla final: ", len(tabla_final_pd))
print("Filas tm:    ", len(df_tm))

---
## 3. Limpieza y Merge

A partir de la tabla unificada de jugadores se aplican reglas de limpieza y validación antes de escribir el resultado final en el lakehouse.

- `limpiar_tabla` — resuelve conflictos entre columnas de distintas fuentes aplicando un orden de prioridad (`tiebreaker`).
- `validar_tabla` — detecta registros con problemas y los separa en una tabla de errores.
- Las filas correctas se escriben en `players_merge` y los errores en `players_errores`.

In [ ]:
import importlib
import pandas as pd
import subprocess
from pyspark.sql import types as T

subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import plata
importlib.reload(plata)
from plata import limpiar_tabla, validar_tabla, pandas_to_spark

df = spark.table("players.mock.players_unified").toPandas()

grupos = [
    {"col_final": "name", "fuentes": ["tm_name", "ts_strPlayer", "fb_player"], "tiebreaker": "tm_name"},
    {"col_final": "team", "fuentes": ["tm_club_name", "ts_strTeam", "fb_team"], "tiebreaker": "tm_club_name"},
]

df_limpio = limpiar_tabla(df, grupos=grupos)
df_validado, errores = validar_tabla(df_limpio)

# Escribir df_validado
df_validado_spark = pandas_to_spark(spark, df_validado)
df_validado_spark.writeTo("players.mock.players_merge") \
    .using("iceberg").createOrReplace()
print("Tabla merge creada.")

# Escribir errores
errores_spark = pandas_to_spark(spark, errores)
errores_spark.writeTo("players.mock.players_errores") \
    .using("iceberg").createOrReplace()
print("Tabla de errores creada.")

## 4. Consultas

### Top goleadores

In [ ]:
spark.sql("""
SELECT
    name,
    team,
    fb_standard_gls
FROM players.mock.players_merge
WHERE fb_standard_gls IS NOT NULL
    AND NOT isnan(fb_standard_gls)
ORDER BY fb_standard_gls DESC
LIMIT 10
""").show()

### Jugadores con más contribuciones (goles + asistencias)

In [ ]:
spark.sql("""
SELECT
    name,
    team,
    fb_standard_gls,
    fb_performance_ast,
    (fb_standard_gls + fb_performance_ast) AS contributions
FROM players.mock.players_merge
WHERE fb_standard_gls IS NOT NULL
    AND NOT isnan(fb_standard_gls)
    AND fb_performance_ast IS NOT NULL
    AND NOT isnan(fb_performance_ast)
ORDER BY contributions DESC
LIMIT 10
""").show()

### Más goles por 90 minutos

In [ ]:
spark.sql("""
SELECT
    name,
    team,
    fb_per_90_minutes_gls
FROM players.mock.players_merge
WHERE fb_per_90_minutes_gls IS NOT NULL
    AND NOT isnan(fb_per_90_minutes_gls)
    AND fb_playing_time_90s IS NOT NULL
    AND NOT isnan(fb_playing_time_90s)
    AND fb_playing_time_90s > 5
ORDER BY fb_per_90_minutes_gls DESC
LIMIT 10
""").show()

### Nacionalidad mas común

In [ ]:
spark.sql("""
SELECT
    ts_strNationality,
    COUNT(*) AS total_players
FROM players.mock.players_merge
WHERE ts_strNationality IS NOT NULL
    AND ts_strNationality != ''
GROUP BY ts_strNationality
ORDER BY total_players DESC
LIMIT 10
""").show()

### Jugadores con redes sociales

In [ ]:
spark.sql("""
SELECT
    name,
    ts_strInstagram,
    ts_strTwitter
FROM players.mock.players_merge
WHERE (ts_strInstagram IS NOT NULL AND ts_strInstagram != '')
    OR (ts_strTwitter IS NOT NULL AND ts_strTwitter != '')
LIMIT 20
""").show(truncate=False)